`conda activate scenicplus`

In [1]:
import os

import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import os
from matplotlib import rcParams
import matplotlib.pyplot as plt

import seaborn as sns

In [2]:
import scatac_fragment_tools

In [3]:
adata = sc.read('/projects/0/einf2548/cruiz/dmg/notebooks/scATAC/data/Mannens2024/Pool_peaks_subset.h5ad')
adata

/home/cruiz2/miniconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/__init__.py:51: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


AnnData object with n_obs × n_vars = 180724 × 410863
    obs: 'Age', 'Agetext', 'Ageunit', 'All_fc_analysis_id', 'Analysis', 'Chemistry', 'Class', 'ClusterName', 'Clusters', 'Clusters_main', 'Cmobarcodes', 'Comment', 'Datecaptured', 'Donor', 'DoubletFinderFlag', 'DoubletFinderScore', 'Editat', 'Editby', 'FRIP', 'FRtss', 'Fragments_in_peaks', 'GA_colsum', 'GA_pooled_colsum', 'Id', 'Label', 'Method', 'NBins', 'NPeaks', 'Name', 'Neuronprop', 'Numpooledanimals', 'Outliers', 'Plugdate', 'Project', 'PseudoAge', 'Roi', 'SEX', 'Sampleok', 'Sex', 'Shortname', 'Species', 'Split', 'Splits', 'Strain', 'SubClusters', 'TSS_fragments', 'Tissue', 'Transcriptome', 'Y', 'barcode', 'chimeric', 'duplicate', 'excluded_reason', 'is__cell_barcode', 'lowmapq', 'mitochondrial', 'passed_filters', 'peak_region_cutsites', 'peak_region_fragments', 'preClusters', 'regions', 'total', 'unmapped', 'Coarse'
    var: 'Annotation', 'Chr', 'CpG%', 'Detailed Annotation', 'Distance to TSS', 'End', 'Entrez ID', 'Focus Ratio-

In [4]:
adata.obs['Coarse'].value_counts()

Endothelial    10000
Neuron         10000
Glioblast      10000
Mural          10000
Microglia      10000
Neuron_Gly     10000
Neuron_HB      10000
Neuron_Int     10000
Neuron_CB      10000
PreOPC         10000
RG             10000
Neuron_Tel     10000
Neuron_Purk    10000
Neuron_MB      10000
Floor_plate     8402
IPC             8350
Roof_plate      7178
OPC             6262
Neuron_GABA     3198
Neuron_DA       2288
Schwann         1836
COP             1159
Neuron_DRG      1019
PVM              784
Lymphoid         248
Name: Coarse, dtype: int64

In [5]:
adata.obs['Class'].value_counts()

Neuron         94855
Radial_glia    42466
Oligo          12371
Immune         11032
Vascular       10597
Fibroblast      9403
Name: Class, dtype: int64

In [6]:
adata.obs['ClusterName'].value_counts()

Floor_plate         8402
VZ_prog             7636
Roof_plate          7178
PreOPC              6886
Neur_Int_LGE_CGE    5429
                    ... 
Gbl_HB_3             135
VLMC_MGP_1           122
End_early_2           92
Gbl_late              73
VLMC_SLC6A13_3        53
Name: ClusterName, Length: 135, dtype: int64

In [7]:
# Define the maximum number of cells per cell type
max_cells_per_celltype = 1000

# Convert adata.obs to a DataFrame and include the 'Coarse' column
metadata_df = adata.obs.copy()

# Add a barcode or identifier column if not already present
metadata_df['barcode'] = metadata_df.index

# Group by the 'Coarse' column and sample cells
sampled_metadata = (metadata_df
                    .groupby('Coarse')
                    .apply(lambda x: x.sample(n=min(len(x), max_cells_per_celltype), random_state=42))
                    .reset_index(drop=True))

# Subset the AnnData object with the sampled barcodes
adata_subset = adata[sampled_metadata['barcode'], :].copy()
adata_subset

AnnData object with n_obs × n_vars = 24032 × 410863
    obs: 'Age', 'Agetext', 'Ageunit', 'All_fc_analysis_id', 'Analysis', 'Chemistry', 'Class', 'ClusterName', 'Clusters', 'Clusters_main', 'Cmobarcodes', 'Comment', 'Datecaptured', 'Donor', 'DoubletFinderFlag', 'DoubletFinderScore', 'Editat', 'Editby', 'FRIP', 'FRtss', 'Fragments_in_peaks', 'GA_colsum', 'GA_pooled_colsum', 'Id', 'Label', 'Method', 'NBins', 'NPeaks', 'Name', 'Neuronprop', 'Numpooledanimals', 'Outliers', 'Plugdate', 'Project', 'PseudoAge', 'Roi', 'SEX', 'Sampleok', 'Sex', 'Shortname', 'Species', 'Split', 'Splits', 'Strain', 'SubClusters', 'TSS_fragments', 'Tissue', 'Transcriptome', 'Y', 'barcode', 'chimeric', 'duplicate', 'excluded_reason', 'is__cell_barcode', 'lowmapq', 'mitochondrial', 'passed_filters', 'peak_region_cutsites', 'peak_region_fragments', 'preClusters', 'regions', 'total', 'unmapped', 'Coarse'
    var: 'Annotation', 'Chr', 'CpG%', 'Detailed Annotation', 'Distance to TSS', 'End', 'Entrez ID', 'Focus Ratio-R

In [10]:
adata_subset.obs['Coarse'].value_counts()

COP            1000
Endothelial    1000
Floor_plate    1000
Glioblast      1000
IPC            1000
Microglia      1000
Mural          1000
Neuron         1000
Neuron_CB      1000
Neuron_Purk    1000
Neuron_DA      1000
Neuron_DRG     1000
Neuron_GABA    1000
Neuron_Gly     1000
Neuron_HB      1000
Neuron_Int     1000
Neuron_MB      1000
Roof_plate     1000
Neuron_Tel     1000
OPC            1000
PreOPC         1000
Schwann        1000
RG             1000
PVM             784
Lymphoid        248
Name: Coarse, dtype: int64

In [11]:
# Create the data as a dictionary
data = {
    'sample': ['fetal'],
    'path_to_fragment_file': ['/projects/0/einf2548/cruiz/dmg/public_data/Mannens2024/compiled_fragments/compiled_filtered_fragments.tsv.sorted.gz']
}

# Convert the dictionary to a DataFrame
df = pd.DataFrame(data)
df

# Save the DataFrame to a TSV file
df.to_csv('data/Mannens2024/sample_to_fragment.tsv', sep='\t', index=False)

print("TSV file 'sample_to_fragment.tsv' has been created.")

TSV file 'sample_to_fragment.tsv' has been created.


In [12]:
df

,sample,path_to_fragment_file
0,fetal,/projects/0/einf2548/cruiz/dmg/public_data/Man...


In [13]:
import pandas as pd

# Assuming `adata` is your AnnData object
sample = 'fetal'

# Extract cell_type from adata.obs['Coarse']
cell_types = adata_subset.obs['Coarse']

# Extract cell barcodes from adata.obs_names
cell_barcodes = adata_subset.obs_names+'-1'

# Create a DataFrame for the required information
df = pd.DataFrame({
    'sample': sample,
    'cell_type': cell_types,
    'cell_barcode': cell_barcodes
})

# Save the DataFrame to a TSV file
df.to_csv('data/Mannens2024/cell_type_to_cell_barcode.tsv', sep='\t', index=False)

print("TSV file 'cell_type_to_cell_barcode.tsv' has been created.")

TSV file 'cell_type_to_cell_barcode.tsv' has been created.


In [14]:
df

,sample,cell_type,cell_barcode
CellID,,,
10X242_2:TACGCAATCCATGTTT,fetal,COP,10X242_2:TACGCAATCCATGTTT-1
10X315_4:TACGCCTAGCTAGCAG,fetal,COP,10X315_4:TACGCCTAGCTAGCAG-1
10X315_4:TGCTTCGAGAGCCACA,fetal,COP,10X315_4:TGCTTCGAGAGCCACA-1
10X347_3:CCAATGAAGTAGCAAT,fetal,COP,10X347_3:CCAATGAAGTAGCAAT-1
10X369_3:AAACTCGGTATTCGAC,fetal,COP,10X369_3:AAACTCGGTATTCGAC-1
...,...,...,...
10X250_4:TTTGTGTAGAGATTAC,fetal,Schwann,10X250_4:TTTGTGTAGAGATTAC-1
10X315_4:GATCGTAAGCCCATTA,fetal,Schwann,10X315_4:GATCGTAAGCCCATTA-1
10X406_5:TCAACAATCTAATCCT,fetal,Schwann,10X406_5:TCAACAATCTAATCCT-1


In [15]:
df['cell_type'].value_counts()

COP            1000
Endothelial    1000
Floor_plate    1000
Glioblast      1000
IPC            1000
Microglia      1000
Mural          1000
Neuron         1000
Neuron_CB      1000
Neuron_Purk    1000
Neuron_DA      1000
Neuron_DRG     1000
Neuron_GABA    1000
Neuron_Gly     1000
Neuron_HB      1000
Neuron_Int     1000
Neuron_MB      1000
Roof_plate     1000
Neuron_Tel     1000
OPC            1000
PreOPC         1000
Schwann        1000
RG             1000
PVM             784
Lymphoid        248
Name: cell_type, dtype: int64

In [ ]:
!scatac_fragment_tools split \
    -f /projects/0/einf2548/cruiz/dmg/notebooks/scATAC/data/Mannens2024/sample_to_fragment.tsv \
    -b /projects/0/einf2548/cruiz/dmg/notebooks/scATAC/data/Mannens2024/cell_type_to_cell_barcode.tsv \
    -c /projects/0/einf2548/cruiz/hg38.chrom.sizes \
    -o /projects/0/einf2548/cruiz/dmg/notebooks/scATAC/pycistopic_fetal/pseudobulk_bed_files \
    -n 32 \
    -v

Splitting fragments ...
Processing contig chr1
Processing contig chr10
Skipping contig chr10_GL383545v1_alt because it is not in the fragments file
Skipping contig chr10_GL383546v1_alt because it is not in the fragments file
Skipping contig chr10_KI270824v1_alt because it is not in the fragments file
Skipping contig chr10_KI270825v1_alt because it is not in the fragments file
Processing contig chr11
Skipping contig chr11_GL383547v1_alt because it is not in the fragments file
Skipping contig chr11_JH159136v1_alt because it is not in the fragments file
Skipping contig chr11_JH159137v1_alt because it is not in the fragments file
Skipping contig chr11_KI270721v1_random because it is not in the fragments file
Skipping contig chr11_KI270826v1_alt because it is not in the fragments file
Skipping contig chr11_KI270827v1_alt because it is not in the fragments file
Skipping contig chr11_KI270829v1_alt because it is not in the fragments file
Skipping contig chr11_KI270830v1_alt because it is not 

In [1]:
!scatac_fragment_tools split \
    -f /projects/0/einf2548/cruiz/dmg/notebooks/scATAC/data/Mannens2024/sample_to_fragment.tsv \
    -b /projects/0/einf2548/cruiz/dmg/notebooks/scATAC/data/Mannens2024/cell_type_to_cell_barcode.tsv \
    -c /projects/0/einf2548/cruiz/hg38.chrom.sizes \
    -o /projects/0/einf2548/cruiz/dmg/notebooks/scATAC/pycistopic_fetal/pseudobulk_bed_files \
    -n 32 \
    -v

Splitting fragments ...
Processing contig chr1
Processing contig chr10
Skipping contig chr10_GL383545v1_alt because it is not in the fragments file
Skipping contig chr10_GL383546v1_alt because it is not in the fragments file
Skipping contig chr10_KI270824v1_alt because it is not in the fragments file
Skipping contig chr10_KI270825v1_alt because it is not in the fragments file
Processing contig chr11
Skipping contig chr11_GL383547v1_alt because it is not in the fragments file
Skipping contig chr11_JH159136v1_alt because it is not in the fragments file
Skipping contig chr11_JH159137v1_alt because it is not in the fragments file
Skipping contig chr11_KI270721v1_random because it is not in the fragments file
Skipping contig chr11_KI270826v1_alt because it is not in the fragments file
Skipping contig chr11_KI270827v1_alt because it is not in the fragments file
Skipping contig chr11_KI270829v1_alt because it is not in the fragments file
Skipping contig chr11_KI270830v1_alt because it is not 